How can I use a Logistic Regression with a regression problem?

I added to our dataset a column indicating whether the Linear Regression 4MA overpredicted (1) or underpredicted (0) the price, then set that as the target for a Logistic Regression. So the LogReg is predicting the LinReg's performance.


# Could I also do this with more classification models? And for each one, go by each regression model to sum up the results?

In [3]:
import pandas as pd

In [5]:
linreg_results = pd.read_csv('/Users/matthewcavanaugh/Desktop/GitHub/magnimind/Auto Scout Project/Model Results/Step 5 - Linear Regression Comparison.csv')
linreg_results

,4MA Error,4MA Squared Error,4MA Absolute Error
0,-1458.25,2131286,1458.25
1,5188.00,26924205,5188.00
2,855.25,737655,855.25
3,-1470.50,2177079,1470.50
4,-768.75,591259,768.75
...,...,...,...
15095,-4389.50,19267724,4389.50
15096,-526.25,277631,526.25
15097,-769.00,682426,769.00
15098,-5979.50,35756880,5979.50


In [6]:
linreg_results['LinReg Overpredicted'] = (linreg_results['4MA Error'] > 0).astype(int)
linreg_results

,4MA Error,4MA Squared Error,4MA Absolute Error,LinReg Overpredicted
0,-1458.25,2131286,1458.25,0
1,5188.00,26924205,5188.00,1
2,855.25,737655,855.25,1
3,-1470.50,2177079,1470.50,0
4,-768.75,591259,768.75,0
...,...,...,...,...
15095,-4389.50,19267724,4389.50,0
15096,-526.25,277631,526.25,0
15097,-769.00,682426,769.00,0
15098,-5979.50,35756880,5979.50,0


In [7]:
linreg_results[linreg_results['4MA Absolute Error'] == 0]

,4MA Error,4MA Squared Error,4MA Absolute Error,LinReg Overpredicted


There were no ties, so every prediction is either over or under.

In [8]:
scout_car = pd.read_csv('Step 4 - Final Dataset with dummies.csv')
scout_car['LinReg Overpredicted'] = linreg_results['LinReg Overpredicted']
scout_car

,Price,KM,Registration,Horsepower (kW),Previous Owners,Inspection new,Warranty (months),# of Doors,# of Seats,Displacement (cc),...,Country version_Spain,Upholstery Material_Full leather,Upholstery Material_Other,Upholstery Material_Part leather,Upholstery Material_Velour,Upholstery Material_alcantara,Upholstery Color_Brown,Upholstery Color_Grey,Upholstery Color_Other,LinReg Overpredicted
0,15770.0,56013.0,2016.0,66.0,2,1,0.0,5.0,5.0,1422.0,...,0,0,0,0,0,0,0,0,0,0
1,14500.0,80000.0,2017.0,141.0,0,0,0.0,3.0,4.0,1798.0,...,0,0,0,0,0,0,0,1,0,1
2,14640.0,83450.0,2016.0,85.0,1,0,0.0,5.0,4.0,1598.0,...,0,0,0,0,0,0,0,0,0,1
3,14500.0,73000.0,2016.0,66.0,1,0,0.0,3.0,4.0,1422.0,...,0,0,0,0,0,0,0,0,0,0
4,16790.0,16200.0,2016.0,66.0,1,1,0.0,5.0,5.0,1422.0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15095,39950.0,10.0,2016.0,147.0,0,0,0.0,5.0,5.0,1997.0,...,0,1,0,0,0,0,0,0,0,0
15096,39885.0,9900.0,2019.0,165.0,1,0,0.0,5.0,5.0,1798.0,...,0,0,0,0,0,0,0,0,0,0
15097,39875.0,15.0,2019.0,146.0,1,1,0.0,5.0,7.0,1997.0,...,0,1,0,0,0,0,0,0,0,0
15098,39700.0,10.0,2019.0,147.0,0,0,0.0,5.0,7.0,1997.0,...,0,0,0,1,0,0,0,0,0,0


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score
from sklearn.metrics import median_absolute_error
from sklearn.model_selection import GridSearchCV
from scipy import stats

# Features
X = scout_car.drop(columns=['LinReg Overpredicted'])

# Target
y = scout_car['LinReg Overpredicted']

# Split into training and testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

logreg = LogisticRegression()
logreg.fit(X_train, y_train)


from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import KFold, cross_val_score


kf = KFold(n_splits=5, shuffle=True, random_state=42)
y_pred_all = cross_val_predict(logreg, X, y, cv=kf)

/opt/anaconda3/envs/IOD2/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/opt/anaconda3/envs/IOD2/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logisti

In [10]:
print(len(y_pred_all), len(X))

15100 15100


In [11]:
y_pred = logreg.predict(X_test)
y_pred

array([0, 1, 0, ..., 0, 0, 1])

In [12]:
scout_car['LogReg Prediction'] = y_pred_all
scout_car['LogReg Correct'] = (scout_car['LinReg Overpredicted'] == scout_car['LogReg Prediction']).astype(int)
scout_car

,Price,KM,Registration,Horsepower (kW),Previous Owners,Inspection new,Warranty (months),# of Doors,# of Seats,Displacement (cc),...,Upholstery Material_Other,Upholstery Material_Part leather,Upholstery Material_Velour,Upholstery Material_alcantara,Upholstery Color_Brown,Upholstery Color_Grey,Upholstery Color_Other,LinReg Overpredicted,LogReg Prediction,LogReg Correct
0,15770.0,56013.0,2016.0,66.0,2,1,0.0,5.0,5.0,1422.0,...,0,0,0,0,0,0,0,0,0,1
1,14500.0,80000.0,2017.0,141.0,0,0,0.0,3.0,4.0,1798.0,...,0,0,0,0,0,1,0,1,1,1
2,14640.0,83450.0,2016.0,85.0,1,0,0.0,5.0,4.0,1598.0,...,0,0,0,0,0,0,0,1,0,0
3,14500.0,73000.0,2016.0,66.0,1,0,0.0,3.0,4.0,1422.0,...,0,0,0,0,0,0,0,0,0,1
4,16790.0,16200.0,2016.0,66.0,1,1,0.0,5.0,5.0,1422.0,...,0,0,0,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15095,39950.0,10.0,2016.0,147.0,0,0,0.0,5.0,5.0,1997.0,...,0,0,0,0,0,0,0,0,0,1
15096,39885.0,9900.0,2019.0,165.0,1,0,0.0,5.0,5.0,1798.0,...,0,0,0,0,0,0,0,0,0,1
15097,39875.0,15.0,2019.0,146.0,1,1,0.0,5.0,7.0,1997.0,...,0,0,0,0,0,0,0,0,0,1
15098,39700.0,10.0,2019.0,147.0,0,0,0.0,5.0,7.0,1997.0,...,0,1,0,0,0,0,0,0,0,1


In [13]:
scout_car['LogReg Correct'].value_counts()

LogReg Correct
1    10068
0     5032
Name: count, dtype: int64

Confusion Matrix?

Rank each column by correlation. This list will show how well the Logistic Regression was able to predict the Linear Regression's accuracy. I am curious whether similar columns are tripping up both models.

In [16]:
# Get correlations of all columns with 'LogReg Correct'
corrs = scout_car.corr()['LogReg Correct']

# Drop self-correlation if present
corrs = corrs.drop('LogReg Correct')

# Show all rows when printing
pd.set_option('display.max_rows', None)

# Sort by absolute correlation (strongest to weakest)
corrs_sorted = corrs.abs().sort_values(ascending=False)

corrs_sorted

LinReg Overpredicted                     0.094247
Type_Used                                0.077567
Registration                             0.064083
Type_New                                 0.062046
CO2 Emission (g CO2/km (comb))           0.058119
make_model_Audi A3                       0.055111
Previous Owners                          0.054485
Price                                    0.054037
Horsepower (kW)                          0.051437
Alarm system                             0.050461
Parking assist system sensors rear       0.050403
make_model_Opel Insignia                 0.050368
make_model_Opel Astra                    0.048017
Traffic sign recognition                 0.044922
Blind spot monitor                       0.044400
Upholstery Material_Other                0.044068
Split rear seats                         0.042354
Upholstery Material_Part leather         0.041127
Head airbag                              0.040464
Tire pressure monitoring system          0.040307


In [41]:
# Compute correlations
corrs = scout_car.corr()['LogReg Correct'].drop('LogReg Correct')

# Convert to DataFrame
ranked_corrs_df = corrs.reset_index()
ranked_corrs_df.columns = ['Column', 'Correlation']

# Add absolute correlation for ranking
ranked_corrs_df['AbsCorrelation'] = ranked_corrs_df['Correlation'].abs()

# Rank (1 = strongest)
ranked_corrs_df['Correlation Rank'] = ranked_corrs_df['Correlation'].rank(
    method='first', ascending=False
).astype(int)

# Rank (1 = strongest)
ranked_corrs_df['AbsCorrelation Rank'] = ranked_corrs_df['AbsCorrelation'].rank(
    method='first', ascending=False
).astype(int)

# Sort by rank
ranked_corrs_df = ranked_corrs_df.sort_values('AbsCorrelation Rank').reset_index(drop=True)

# Show full list
import pandas as pd
pd.reset_option('display.max_rows', None)

ranked_corrs_df

,Column,Correlation,AbsCorrelation,Correlation Rank,AbsCorrelation Rank
0,LinReg Overpredicted,0.094247,0.094247,1,1
1,Type_Used,-0.077567,0.077567,174,2
2,Registration,0.064083,0.064083,2,3
3,Type_New,0.062046,0.062046,3,4
4,CO2 Emission (g CO2/km (comb)),0.058119,0.058119,4,5
...,...,...,...,...,...
169,Emission Class_Euro 6,-0.001007,0.001007,79,170
170,Body Color_Red,0.000741,0.000741,75,171
171,Emission Class_Euro 6d-TEMP,-0.000643,0.000643,78,172
172,Lumbar support,0.000248,0.000248,76,173


In [20]:
ranked_corrs_df.to_csv('/Users/matthewcavanaugh/Desktop/GitHub/magnimind/Auto Scout Project/Model Results/Correlation Rankings LogReg.csv', index=False)